# P1-E16.7C {renamed P1-E11.C} Leakage-Safe Endpoint Selection, Cross-Fitted Reduced Fusion, and Deployment-Specification Freeze

## Purpose

This experiment converts the authenticated development out-of-fold evidence from P1-E16.7B-R into one prespecified deployment decision for each of four endpoints: Normal versus AD, MCI versus AD, Normal versus MCI, and three-class Normal/MCI/AD classification.

It answers the reviewers' remaining development-selection concerns by:

1. evaluating reduced fusion only from base predictions generated without the evaluated recording;
2. selecting fusion components and weights without using the held-out outer fold;
3. comparing candidates through paired recording-level bootstrap differences;
4. controlling multiplicity within each endpoint using Holm adjustment;
5. applying a deterministic selection and tie rule; and
6. freezing machine-readable deployment specifications and hashes.

**Evidence boundary:** this notebook uses development out-of-fold predictions only. It does not load the official CAUEEG test partition or either external cohort. Full-development base-model refitting and one-time official-test evaluation belong to P1-E16.7D.

## 1. Configuration, paths, and scientific contract


In [ ]:
from pathlib import Path
import hashlib, json, platform, sys
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import balanced_accuracy_score, accuracy_score, f1_score, matthews_corrcoef, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone
import joblib

SEED=20261010
N_BOOT=5000
TIE_TOLERANCE=0.005
OUTER_FOLDS=5
TASKS={
    'normal_vs_ad':['Normal','AD'],
    'mci_vs_ad':['MCI','AD'],
    'normal_vs_mci':['Normal','MCI'],
    'three_class':['Normal','MCI','AD'],
}
METHOD_COMPLEXITY={
    'Conventional nested benchmark':1,
    'CNN–BiLSTM feature fusion':2,
    'Functional-connectivity GNN':2,
    'CEEDNet adaptation':2,
    'SSL multiresolution attention':2,
    'Cross-fitted top-2 fusion':3,
    'Cross-fitted top-3 fusion':4,
}

PROJECT_ROOT=Path(r'D:\EEG_AD_Research\dementia\paper one')
OUTPUTS_ROOT=PROJECT_ROOT/'paper1_outputs'
E16_7B_ROOT=OUTPUTS_ROOT/'p1_e16_7b_r_unified_four_endpoint_nested_method_comparison'
OUTPUT_ROOT=OUTPUTS_ROOT/'p1_e16_7c_endpoint_selection_cross_fitted_reduced_fusion_freeze'
TABLES=OUTPUT_ROOT/'tables'; FIGURES=OUTPUT_ROOT/'figures'; MODELS=OUTPUT_ROOT/'models'; CONTRACTS=OUTPUT_ROOT/'contracts'
for p in (TABLES,FIGURES,MODELS,CONTRACTS): p.mkdir(parents=True,exist_ok=True)

EXACT_PREDICTIONS=E16_7B_ROOT/'tables'/'08_ALL_METHOD_NESTED_OOF_PREDICTIONS.csv'
EXACT_SUMMARY=E16_7B_ROOT/'tables'/'09_METHOD_PERFORMANCE_SUMMARY.csv'
EXACT_FOLD_METRICS=E16_7B_ROOT/'tables'/'10_OUTER_FOLD_METHOD_METRICS.csv'
EXACT_INTEGRITY=E16_7B_ROOT/'tables'/'12_FINAL_INTEGRITY_CHECKS.csv'
EXACT_RUN_RECORD=E16_7B_ROOT/'run_record.json'

def sha256_file(path,chunk=1<<20):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        while True:
            b=f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()
def save_table(frame,name):
    path=TABLES/f'{name}.csv'; frame.to_csv(path,index=False); return path
def json_default(x):
    if isinstance(x,(np.integer,)): return int(x)
    if isinstance(x,(np.floating,)): return float(x)
    if isinstance(x,np.ndarray): return x.tolist()
    if isinstance(x,Path): return str(x)
    raise TypeError(type(x).__name__)
def holm_adjust(pvalues):
    p=np.asarray(pvalues,float); order=np.argsort(p); adjusted=np.empty(len(p),float); running=0.0
    for rank,idx in enumerate(order):
        running=max(running,(len(p)-rank)*p[idx]); adjusted[idx]=min(1.0,running)
    return adjusted
print({'python':sys.version.split()[0],'platform':platform.platform(),'output_root':str(OUTPUT_ROOT)})

## 2. Authenticate the completed P1-E16.7B-R evidence lock


In [ ]:
required=[EXACT_PREDICTIONS,EXACT_SUMMARY,EXACT_FOLD_METRICS,EXACT_INTEGRITY,EXACT_RUN_RECORD]
missing=[str(p) for p in required if not p.is_file()]
if missing: raise FileNotFoundError('Missing completed P1-E16.7B-R artifact(s):\n'+'\n'.join(missing))

integrity=pd.read_csv(EXACT_INTEGRITY)
if 'pass' not in integrity or not integrity['pass'].astype(str).str.lower().eq('true').all():
    raise AssertionError('P1-E16.7B-R did not pass every integrity check.')
run7b=json.loads(EXACT_RUN_RECORD.read_text(encoding='utf-8'))
if run7b.get('status')!='PASSED' or not run7b.get('integrity_passed',False):
    raise AssertionError('P1-E16.7B-R run record is not a passed lock.')
if run7b.get('official_test_accessed') or run7b.get('external_data_accessed'):
    raise AssertionError('Development-selection evidence is contaminated by test/external access.')

pred=pd.read_csv(EXACT_PREDICTIONS,dtype={'recording_id':str})
pred['recording_id']=pred.recording_id.astype(str).str.strip().str.zfill(5)
pred=pred.rename(columns={'endpoint':'task','fold':'outer_fold'})
needed={'task','method','outer_fold','recording_id','true_label','predicted_label'}
if not needed.issubset(pred): raise KeyError(f'Prediction table lacks {needed-set(pred)}')
if set(pred.task)!=set(TASKS): raise AssertionError('The four endpoint set is incomplete.')
if pred.duplicated(['task','method','recording_id']).any(): raise AssertionError('Duplicate method-level OOF predictions detected.')

inputs=pd.DataFrame([{'role':p.name,'path':str(p),'sha256':sha256_file(p)} for p in required])
save_table(inputs,'01_AUTHENTICATED_INPUTS'); display(inputs)

## 3. Cross-fitted reduced fusion

For every held-out outer fold, candidate component methods are ranked using only the other four folds. The top two and top three methods form reduced candidate sets. A logistic meta-model is trained on cross-fitted base probabilities from those four folds and predicts only the untouched fifth fold. Consequently, neither component selection nor fusion fitting uses the evaluated recording.

Probability features are transformed as $\log(p+\epsilon)$ and standardized inside the meta-training data. The fixed regularization value $C=1$ avoids an additional small-sample tuning layer.

In [ ]:
BASE_METHODS=sorted(pred.method.unique())
if len(BASE_METHODS)<3: raise AssertionError('At least three complete base methods are required for reduced fusion.')

def probability_columns(classes): return [f'probability_{c}' for c in classes]
def ba_for(g):
    return balanced_accuracy_score(g.true_label,g.predicted_label)
def fusion_features(frame,methods,classes):
    blocks=[]; order=None
    for method in methods:
        q=frame.loc[frame.method.eq(method)].sort_values('recording_id')
        ids=q.recording_id.to_numpy()
        if order is None: order=ids
        elif not np.array_equal(order,ids): raise AssertionError('Base predictions are not recording-aligned.')
        p=np.clip(q[probability_columns(classes)].to_numpy(float),1e-7,1-1e-7)
        blocks.append(np.log(p))
    return order,np.hstack(blocks)

fusion_rows=[]; fusion_contract_rows=[]
for task,classes in TASKS.items():
    qtask=pred.loc[pred.task.eq(task)].copy()
    for holdout in range(1,OUTER_FOLDS+1):
        train=qtask.loc[qtask.outer_fold.ne(holdout)]
        test=qtask.loc[qtask.outer_fold.eq(holdout)]
        ranking=(train.groupby('method').apply(ba_for)
                 .rename('training_fold_pooled_ba').reset_index()
                 .assign(complexity=lambda x:x.method.map(METHOD_COMPLEXITY))
                 .sort_values(['training_fold_pooled_ba','complexity','method'],ascending=[False,True,True]))
        for k,label in [(2,'Cross-fitted top-2 fusion'),(3,'Cross-fitted top-3 fusion')]:
            methods=ranking.head(k).method.tolist()
            train_ids,Xtr=fusion_features(train,methods,classes)
            test_ids,Xte=fusion_features(test,methods,classes)
            truth_train=(train.loc[train.method.eq(methods[0])].sort_values('recording_id')
                         .true_label.map({c:i for i,c in enumerate(classes)}).to_numpy(int))
            truth_test_frame=test.loc[test.method.eq(methods[0])].sort_values('recording_id')
            truth_test=truth_test_frame.true_label.map({c:i for i,c in enumerate(classes)}).to_numpy(int)
            model=Pipeline([('scale',StandardScaler()),('meta',LogisticRegression(
                C=1.0,max_iter=3000,class_weight='balanced',random_state=SEED+holdout))])
            model.fit(Xtr,truth_train); prob=model.predict_proba(Xte); predicted=prob.argmax(1)
            for i,rid in enumerate(test_ids):
                row={'task':task,'outer_fold':holdout,'recording_id':rid,'true_label':classes[truth_test[i]],
                     'predicted_label':classes[predicted[i]],'method':label}
                row.update({f'probability_{c}':float(prob[i,j]) for j,c in enumerate(classes)})
                fusion_rows.append(row)
            fusion_contract_rows.append({'task':task,'outer_fold':holdout,'fusion':label,
                                         'components':' | '.join(methods),'meta_C':1.0,
                                         'component_ranking_source':'other four outer folds only'})

fusion_pred=pd.DataFrame(fusion_rows)
fusion_contract=pd.DataFrame(fusion_contract_rows)
candidate_pred=pd.concat([pred,fusion_pred],ignore_index=True,sort=False)
save_table(fusion_pred,'02_CROSS_FITTED_REDUCED_FUSION_OOF_PREDICTIONS')
save_table(fusion_contract,'03_OUTER_FOLD_FUSION_COMPONENT_AUDIT')
display(fusion_contract)

## 4. Candidate performance and paired recording-level bootstrap comparisons


In [ ]:
def metric_row(q,task,method):
    classes=TASKS[task]; c2i={c:i for i,c in enumerate(classes)}
    y=q.true_label.map(c2i).to_numpy(int); yp=q.predicted_label.map(c2i).to_numpy(int)
    p=q[probability_columns(classes)].to_numpy(float)
    auc=roc_auc_score(y,p[:,1]) if len(classes)==2 else roc_auc_score(y,p,multi_class='ovr',average='macro',labels=np.arange(len(classes)))
    fold_ba=[balanced_accuracy_score(g.true_label,g.predicted_label) for _,g in q.groupby('outer_fold')]
    return {'task':task,'method':method,'n':len(q),'balanced_accuracy':balanced_accuracy_score(y,yp),
            'accuracy':accuracy_score(y,yp),'macro_f1':f1_score(y,yp,average='macro'),
            'mcc':matthews_corrcoef(y,yp),'roc_auc':auc,'fold_ba_mean':np.mean(fold_ba),
            'fold_ba_sd':np.std(fold_ba,ddof=1),'fold_ba_se':np.std(fold_ba,ddof=1)/np.sqrt(len(fold_ba)),
            'chance_balanced_accuracy':1/len(classes),'complexity_rank':METHOD_COMPLEXITY[method]}

metrics=pd.DataFrame([metric_row(q,t,m) for (t,m),q in candidate_pred.groupby(['task','method'],sort=False)])

def paired_bootstrap(task,method,reference,rng):
    classes=TASKS[task]
    a=candidate_pred.loc[(candidate_pred.task.eq(task))&(candidate_pred.method.eq(method))].set_index('recording_id').sort_index()
    b=candidate_pred.loc[(candidate_pred.task.eq(task))&(candidate_pred.method.eq(reference))].set_index('recording_id').sort_index()
    if not a.index.equals(b.index) or not a.true_label.equals(b.true_label): raise AssertionError('Paired bootstrap alignment failure.')
    strata={c:np.flatnonzero(a.true_label.to_numpy()==c) for c in classes}; delta=[]
    for _ in range(N_BOOT):
        ix=np.concatenate([rng.choice(strata[c],len(strata[c]),replace=True) for c in classes])
        delta.append(balanced_accuracy_score(a.true_label.to_numpy()[ix],a.predicted_label.to_numpy()[ix])-
                     balanced_accuracy_score(b.true_label.to_numpy()[ix],b.predicted_label.to_numpy()[ix]))
    delta=np.asarray(delta); observed=metrics.loc[(metrics.task.eq(task))&(metrics.method.eq(method)),'balanced_accuracy'].iloc[0]-metrics.loc[(metrics.task.eq(task))&(metrics.method.eq(reference)),'balanced_accuracy'].iloc[0]
    # Two-sided bootstrap sign probability; bounded away from zero for finite resamples.
    p=2*min((np.sum(delta<=0)+1)/(N_BOOT+1),(np.sum(delta>=0)+1)/(N_BOOT+1)); p=min(1.0,p)
    return observed,np.quantile(delta,.025),np.quantile(delta,.975),p

rng=np.random.default_rng(SEED); comparison=[]
for task in TASKS:
    reference='Conventional nested benchmark'
    methods=metrics.loc[(metrics.task.eq(task))&metrics.method.ne(reference),'method'].tolist()
    start=len(comparison)
    for method in methods:
        d,lo,hi,p=paired_bootstrap(task,method,reference,rng)
        comparison.append({'task':task,'method':method,'reference':reference,'paired_ba_difference':d,
                           'difference_ci_low':lo,'difference_ci_high':hi,'unadjusted_p_value':p})
    pvals=[r['unadjusted_p_value'] for r in comparison[start:]]
    adjusted=holm_adjust(pvals)
    for row,padj in zip(comparison[start:],adjusted): row['holm_adjusted_p_value']=padj
comparison=pd.DataFrame(comparison)
save_table(metrics,'04_ALL_CANDIDATE_DEVELOPMENT_METRICS')
save_table(comparison,'05_PAIRED_RECORDING_BOOTSTRAP_VS_CONVENTIONAL')
display(metrics.sort_values(['task','balanced_accuracy'],ascending=[True,False]).round(4))
display(comparison.round(4))

## 5. Prespecified endpoint-specific selection

The selected candidate is the one with the highest pooled development OOF balanced accuracy. If candidates differ by no more than 0.005, the lower-complexity candidate is selected; remaining ties are resolved by higher macro-F1, lower fold standard deviation, higher ROC-AUC, and finally the method name. This rule was fixed before official-test access and does not use a desired performance target.

In [ ]:
selection=[]
for task in TASKS:
    q=metrics.loc[metrics.task.eq(task)].copy()
    best=q.balanced_accuracy.max(); q['within_tie_tolerance']=q.balanced_accuracy.ge(best-TIE_TOLERANCE-1e-12)
    eligible=q.loc[q.within_tie_tolerance].sort_values(
        ['complexity_rank','balanced_accuracy','macro_f1','fold_ba_sd','roc_auc','method'],
        ascending=[True,False,False,True,False,True])
    chosen=eligible.iloc[0]
    selection.append({'task':task,'selected_method':chosen.method,'balanced_accuracy':chosen.balanced_accuracy,
                      'macro_f1':chosen.macro_f1,'roc_auc':chosen.roc_auc,'fold_ba_sd':chosen.fold_ba_sd,
                      'complexity_rank':int(chosen.complexity_rank),'best_observed_ba':best,
                      'selection_rule':'max OOF BA; <=0.005 tie favors lower complexity, then macro-F1, stability, ROC-AUC, name',
                      'official_test_used':False,'external_data_used':False})
selection=pd.DataFrame(selection)
save_table(selection,'06_ENDPOINT_SELECTION_DECISIONS'); display(selection.round(4))

## 6. Freeze full-development deployment specifications


In [ ]:
deployment=[]
for row in selection.itertuples():
    task=row.task; classes=TASKS[task]; selected=row.selected_method
    spec={'endpoint':task,'classes':classes,'selected_method':selected,'selection_metric':'development OOF balanced accuracy',
          'tie_tolerance':TIE_TOLERANCE,'seed':SEED,'official_test_accessed':False,'external_data_accessed':False,
          'source_prediction_sha256':sha256_file(EXACT_PREDICTIONS)}
    if selected.startswith('Cross-fitted'):
        k=2 if 'top-2' in selected else 3
        ranking=(metrics.loc[(metrics.task.eq(task))&metrics.method.isin(BASE_METHODS)]
                 .sort_values(['balanced_accuracy','complexity_rank','method'],ascending=[False,True,True]))
        components=ranking.head(k).method.tolist()
        ids,X=fusion_features(pred.loc[pred.task.eq(task)],components,classes)
        truth=(pred.loc[(pred.task.eq(task))&(pred.method.eq(components[0]))].sort_values('recording_id')
               .true_label.map({c:i for i,c in enumerate(classes)}).to_numpy(int))
        meta=Pipeline([('scale',StandardScaler()),('meta',LogisticRegression(C=1.0,max_iter=3000,class_weight='balanced',random_state=SEED))])
        meta.fit(X,truth); model_path=MODELS/f'{task}__frozen_meta_fusion.joblib'; joblib.dump(meta,model_path)
        spec.update({'components':components,'meta_model_path':str(model_path),'meta_model_sha256':sha256_file(model_path),
                     'meta_feature_order':[f'{m}::{c}::log_probability' for m in components for c in classes]})
    else:
        spec.update({'components':[selected],'base_model_refit_required_in_P1_E16_7D':True})
    contract_path=CONTRACTS/f'{task}__deployment_specification.json'
    contract_path.write_text(json.dumps(spec,indent=2,default=json_default),encoding='utf-8')
    deployment.append({'task':task,'selected_method':selected,'contract_path':str(contract_path),
                       'contract_sha256':sha256_file(contract_path),'requires_base_model_refit':True})
deployment=pd.DataFrame(deployment)
save_table(deployment,'07_FROZEN_DEPLOYMENT_SPECIFICATIONS'); display(deployment)

## 7. Publication-ready selection and uncertainty figures


In [ ]:
labels={'normal_vs_ad':'Normal vs AD','mci_vs_ad':'MCI vs AD','normal_vs_mci':'Normal vs MCI','three_class':'Three class'}
fig,axes=plt.subplots(2,2,figsize=(13,9))
for ax,task in zip(axes.flat,TASKS):
    q=metrics.loc[metrics.task.eq(task)].sort_values('balanced_accuracy')
    colors=['#2e7d32' if m==selection.set_index('task').loc[task,'selected_method'] else '#547aa5' for m in q.method]
    ax.barh(q.method,q.balanced_accuracy,color=colors)
    ax.axvline(1/len(TASKS[task]),ls='--',color='#b22222',label='chance')
    ax.set_title(labels[task]); ax.set_xlabel('Development OOF balanced accuracy'); ax.grid(axis='x',alpha=.25)
fig.suptitle('Endpoint-specific candidate selection after cross-fitted reduced fusion',fontweight='bold')
fig.tight_layout(); fig.savefig(FIGURES/'Figure_P1_E16_7C_Endpoint_Selection.pdf',bbox_inches='tight'); fig.savefig(FIGURES/'Figure_P1_E16_7C_Endpoint_Selection.png',dpi=400,bbox_inches='tight'); plt.show()

fig,axes=plt.subplots(2,2,figsize=(13,9))
for ax,task in zip(axes.flat,TASKS):
    q=comparison.loc[comparison.task.eq(task)].sort_values('paired_ba_difference')
    x=q.paired_ba_difference.to_numpy(); lo=x-q.difference_ci_low.to_numpy(); hi=q.difference_ci_high.to_numpy()-x
    ax.errorbar(x,np.arange(len(q)),xerr=[lo,hi],fmt='o',capsize=3,color='#1f4e79')
    ax.axvline(0,ls='--',color='#444444'); ax.set_yticks(np.arange(len(q))); ax.set_yticklabels(q.method)
    ax.set_title(labels[task]); ax.set_xlabel('Paired BA difference vs conventional (95% bootstrap CI)'); ax.grid(axis='x',alpha=.25)
fig.suptitle('Paired recording-level uncertainty of method differences',fontweight='bold')
fig.tight_layout(); fig.savefig(FIGURES/'Figure_P1_E16_7C_Paired_Differences.pdf',bbox_inches='tight'); fig.savefig(FIGURES/'Figure_P1_E16_7C_Paired_Differences.png',dpi=400,bbox_inches='tight'); plt.show()

## 8. Final integrity and reproducibility lock


In [ ]:
checks=[]
for task,classes in TASKS.items():
    expected=set(pred.loc[pred.task.eq(task),'recording_id'])
    for method in ['Cross-fitted top-2 fusion','Cross-fitted top-3 fusion']:
        q=fusion_pred.loc[(fusion_pred.task.eq(task))&fusion_pred.method.eq(method)]
        p=q[probability_columns(classes)].to_numpy(float)
        checks.extend([
            {'scope':task,'check':f'{method}: one OOF prediction per recording','pass':set(q.recording_id)==expected and not q.recording_id.duplicated().any()},
            {'scope':task,'check':f'{method}: all five folds','pass':set(q.outer_fold.astype(int))==set(range(1,6))},
            {'scope':task,'check':f'{method}: probabilities valid','pass':np.isfinite(p).all() and np.allclose(p.sum(1),1,atol=1e-6)},
            {'scope':task,'check':f'{method}: labels aligned','pass':q.true_label.isin(classes).all()},])
    checks.extend([
        {'scope':task,'check':'exactly one selected deployment specification','pass':selection.task.eq(task).sum()==1 and deployment.task.eq(task).sum()==1},
        {'scope':task,'check':'chance level mathematically correct','pass':np.isclose(metrics.loc[metrics.task.eq(task),'chance_balanced_accuracy'],1/len(classes)).all()},])
checks.extend([
    {'scope':'all','check':'four endpoints frozen','pass':set(selection.task)==set(TASKS)},
    {'scope':'all','check':'official test excluded','pass':True},
    {'scope':'all','check':'external cohorts excluded','pass':True},
    {'scope':'all','check':'performance target excluded from selection','pass':True},
    {'scope':'all','check':'Holm-adjusted comparisons complete','pass':len(comparison)>0 and comparison.holm_adjusted_p_value.notna().all()},])
checks=pd.DataFrame(checks); save_table(checks,'08_FINAL_INTEGRITY_CHECKS'); display(checks)
if not checks['pass'].all(): raise AssertionError('P1-E16.7C integrity failure.')

run={'experiment':'P1-E16.7C endpoint selection, cross-fitted reduced fusion, and deployment-specification freeze',
     'status':'PASSED','completed_utc':datetime.now(timezone.utc).isoformat(),'seed':SEED,'endpoints':TASKS,
     'selected_deployments':selection.to_dict('records'),'official_test_accessed':False,'external_data_accessed':False,
     'performance_target_used':False,'paired_bootstrap_repetitions':N_BOOT,'multiplicity_method':'Holm within endpoint',
     'input_files':inputs.to_dict('records'),'integrity_passed':True}
run_path=OUTPUT_ROOT/'run_record.json'; run_path.write_text(json.dumps(run,indent=2,default=json_default),encoding='utf-8')
print('P1-E16.7C PASSED — ENDPOINT SELECTION, CROSS-FITTED FUSION, AND DEPLOYMENT SPECIFICATIONS FROZEN')